In [ ]:
import os
import re
import argparse
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def parse_training_log(log_path):
    """从 log 文本中提取 Epoch 级别的 Valid 和 Test 指标"""
    if not os.path.exists(log_path):
        raise FileNotFoundError(f"❌ 找不到日志文件: {log_path}")

    records = []
    current_epoch = None
    
    with open(log_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            # 1. 捕捉 Epoch 号
            # 匹配格式: "Ep 12 🌍(GLOBAL!) | T:15.2s..." 或 "Ep 1  ..."
            ep_match = re.search(r'^Ep\s+(\d+)', line)
            if ep_match:
                current_epoch = int(ep_match.group(1))
                
            # 2. 捕捉 VAL 指标
            # 匹配格式: "[VAL ] AUUC:0.8920 | AUQC:0.8950 | L10:0.0082 | AUC:0.9100"
            elif "[VAL ]" in line and current_epoch is not None:
                metrics_str = line.split("[VAL ]")[-1]
                val_record = {"Epoch": current_epoch, "Phase": "Valid"}
                for item in metrics_str.split('|'):
                    if ':' in item:
                        k, v = item.split(':')
                        val_record[k.strip()] = float(v.strip())
                records.append(val_record)
                
            # 3. 捕捉 TEST 指标
            elif "[TEST]" in line and current_epoch is not None:
                metrics_str = line.split("[TEST]")[-1]
                test_record = {"Epoch": current_epoch, "Phase": "Test"}
                for item in metrics_str.split('|'):
                    if ':' in item:
                        k, v = item.split(':')
                        test_record[k.strip()] = float(v.strip())
                records.append(test_record)

    df = pd.DataFrame(records)
    if df.empty:
        print(f"⚠️ 在 {log_path} 中没有解析到任何有效的 [VAL ] / [TEST] 格式的训练指标。")
        
    return df

def plot_learning_curves(df, log_path, target_metrics=['AUUC', 'AUC']):
    """绘制学习曲线并保存"""
    if df.empty: return
    
    # 设置 Seaborn 学术风
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
    
    num_metrics = len(target_metrics)
    fig, axes = plt.subplots(1, num_metrics, figsize=(6 * num_metrics, 5))
    if num_metrics == 1: axes = [axes] # 保证可迭代
    
    for i, metric in enumerate(target_metrics):
        if metric not in df.columns:
            print(f"⚠️ 找不到指标 {metric}，跳过绘图...")
            continue
            
        ax = axes[i]
        sns.lineplot(data=df, x='Epoch', y=metric, hue='Phase', marker='o', ax=ax, 
                     palette={'Valid': '#1f77b4', 'Test': '#ff7f0e'}, linewidth=2)
        
        ax.set_title(f'Learning Curve: {metric}', fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric)
        
        # 🌟 核心：标注最佳 Epoch (基于 Valid 指标)
        valid_df = df[df['Phase'] == 'Valid']
        if not valid_df.empty:
            best_epoch_row = valid_df.loc[valid_df[metric].idxmax()]
            best_ep = best_epoch_row['Epoch']
            best_val = best_epoch_row[metric]
            
            # 画一条垂直虚线指示选出的最佳 Epoch
            ax.axvline(x=best_ep, color='red', linestyle='--', alpha=0.6)
            ax.text(best_ep + 0.5, best_val, f'Best Ep: {int(best_ep)}', color='red', fontsize=10)
            
    plt.tight_layout()
    
    # 自动保存到原日志相同的目录下
    # save_dir = os.path.dirname(log_path)
    # base_name = os.path.splitext(os.path.basename(log_path))[0]
    # save_path = os.path.join(save_dir, f"{base_name}_curves.png")
    plt.show()
    # plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"🎉 成功生成学习曲线对比图，已保存至: {save_path}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Uplift 训练日志学习曲线提取与绘制工具")
    parser.add_argument("--log_path", type=str, required=True, help="要解析的日志文件绝对路径")
    parser.add_argument("--metrics", type=str, nargs='+', default=['AUUC', 'AUC'], help="想要绘制的指标列表，用空格隔开")
    
    args = parser.parse_args()
    
    print(f"🔍 正在解析日志: {args.log_path}")
    df_metrics = parse_training_log(args.log_path)
    
    if not df_metrics.empty:
        # 如果你想看具体的数值，可以取消注释这行把 CSV 也存下来
        # df_metrics.to_csv(args.log_path.replace(".txt", ".csv"), index=False)
        plot_learning_curves(df_metrics, args.log_path, args.metrics)

In [ ]:
import os
import glob
import json
import pandas as pd
import numpy as np

# ==========================================
# 📊 终极聚合器：多种子均值与方差分析引擎
# ==========================================

# 你想重点关注的指标列表 (会自动加上 Test_ 前缀去取)
FOCUS_METRICS = [
    "Target_Y_AUUC",
    "Target_C_AUUC",
    "Target_Y_AUC",
    "Target_Y_AUQC",
    "Target_Y_Lift@10"
]

def parse_all_results(base_dir):
    """递归扫描所有的 final_metrics.json 并解析"""
    # 匹配所有的 final_metrics.json (包括直接在 run_xxx 下的，以及在 seed_xxx 下的)
    search_pattern = os.path.join(base_dir, "**", "final_metrics.json")
    json_files = glob.glob(search_pattern, recursive=True)
    
    if not json_files:
        print(f"⚠️ 在 {base_dir} 下没有找到任何 final_metrics.json 文件！")
        return pd.DataFrame()

    records = []
    for filepath in json_files:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                metrics = json.load(f)
        except Exception as e:
            print(f"⚠️ 跳过损坏的文件: {filepath}")
            continue

        # 解析路径，提取模型信息
        # 路径结构一般是: .../results/criteo/train_y/Model/Version/ExpName/[seed_xxx]/final_metrics.json
        parts = filepath.split('/')
        try:
            # 找到 train_y 的索引
            idx = parts.index("train_y")
            model_name = parts[idx + 1]
            version_name = parts[idx + 2]
            
            # 判断是否有 seed_xxx 文件夹
            parent_dir = parts[-2]
            if "seed_" in parent_dir:
                seed = parent_dir.replace("seed_", "")
            else:
                seed = "Original" # 如果没有 seed_ 文件夹，说明是最初跑的那次
                
        except ValueError:
            model_name = "Unknown"
            version_name = "Unknown"
            seed = "Unknown"

        # 提取核心记录
        record = {
            "Model": model_name,
            "Version": version_name,
            "Seed": seed
        }
        
        for m in FOCUS_METRICS:
            # 默认取 Test 集的结果
            record[m] = metrics.get(f"Test_{m}", np.nan)
            
        records.append(record)

    return pd.DataFrame(records)

def aggregate_and_format(df):
    """计算均值和方差，并格式化为学术界标准的 Mean ± Std"""
    if df.empty:
        return df

    # 1. 按照 Model 和 Version 分组
    grouped = df.groupby(["Model", "Version"])
    
    agg_records = []
    for (model, version), group in grouped:
        row = {
            "Model": model,
            "Version": version,
            "Seeds_Count": len(group) # 统计成功跑完的种子数量
        }
        
        # 2. 对每个指标计算 Mean 和 Std
        for m in FOCUS_METRICS:
            values = group[m].dropna()
            if len(values) == 0:
                row[m] = "N/A"
                # 用于排序的隐藏列
                row[f"{m}_mean_val"] = 0.0 
            else:
                mean_val = values.mean()
                std_val = values.std() if len(values) > 1 else 0.0
                
                # 格式化: 0.9186 ± 0.0012
                # AUUC 和 AUC 保留 4 位小数，Lift 保留 5 位小数
                if "Lift" in m:
                    row[m] = f"{mean_val:.5f} ± {std_val:.5f}"
                else:
                    row[m] = f"{mean_val:.4f} ± {std_val:.4f}"
                    
                row[f"{m}_mean_val"] = mean_val
                
        agg_records.append(row)

    agg_df = pd.DataFrame(agg_records)
    
    # 3. 按照核心指标 Target_Y_AUUC 的均值倒序排列！(选出真正的王者)
    sort_col = "Target_Y_AUUC_mean_val"
    if sort_col in agg_df.columns:
        agg_df = agg_df.sort_values(by=sort_col, ascending=False).reset_index(drop=True)
        # 删掉用来排序的临时隐藏列
        cols_to_drop = [c for c in agg_df.columns if "_mean_val" in c]
        agg_df = agg_df.drop(columns=cols_to_drop)

    return agg_df

if __name__ == "__main__":
    # 你的结果落盘根目录
    base_results_dir = "/NAS/shith/uplift/results/criteo/train_y"
    
    print("🔍 正在疯狂扫描所有实验结果目录...")
    raw_df = parse_all_results(base_results_dir)
    
    if raw_df.empty:
        print("❌ 未提取到任何数据，请检查路径。")
    else:
        print(f"✅ 成功提取到 {len(raw_df)} 条跑完的 JSON 记录。")
        
        # 聚合处理
        final_df = aggregate_and_format(raw_df)
        
        # 为了让终端打印好看，调整 Pandas 显示宽度
        pd.set_option('display.max_rows', 100)
        pd.set_option('display.max_columns', 15)
        pd.set_option('display.width', 200)
        
        print("\n" + "="*120)
        print("🏆 终极 Uplift 性能排行榜 (均值 ± 方差) | 按 Target_Y_AUUC 降序排布")
        print("="*120)
        
        # 打印到控制台
        print(final_df.to_string(index=False))
        
        print("="*120)
        
        # 落盘为 CSV，你可以直接下下来用 Excel 打开，或者粘贴进论文
        out_csv = "final_leaderboard_with_std.csv"
        final_df.to_csv(out_csv, index=False, encoding='utf-8-sig')
        print(f"🎉 统计完毕！请查收您的成绩单: {out_csv}")